In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib
import time 


from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


C:\Users\bilal\AppData\Local\Temp\ipykernel_37236\3297560779.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
df = pd.read_csv("../data/preprocessed_data/news_data_sentiment.csv")

print("Shape :", df.shape)
df.head()


Shape : (675, 17)


,company,ticker,text,created,nid,article_title,article_url,info_index,datetime,datetime_my,date,sentiment,positive_score,negative_score,neutral_score,sentiment_score,n_tokens
0,Petronas Chemicals,5183.KL,PETRONAS Chemicals Group Bhd (KL: PCHEM ) rose...,1789397619000,818088,"Kelington, NuEnergy, GFM, Exsim Hospitality, K...",https://theedgemalaysia.com/node/818088,6,2026-09-14 14:53:39+00:00,2026-09-14 22:53:39+08:00,2026-09-14,positive,0.944639,0.023755,0.031606,0.920885,114
1,Maybank,1155.KL,Malayan Banking Bhd (KL: MAYBANK ) has receive...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,2,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08,positive,0.561400,0.009733,0.428867,0.551667,125
2,Public Bank,1295.KL,Public Bank Bhd (KL: PBBANK ) is proposing to ...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,1,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08,neutral,0.033205,0.019832,0.946963,0.013373,145
3,CIMB,1023.KL,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,1787932174000,816234,"Yinson, Gamuda, WCT, Zetrix AI, HeiTech Padu, ...",https://theedgemalaysia.com/node/816234,6,2026-08-28 15:49:34+00:00,2026-08-28 23:49:34+08:00,2026-08-28,positive,0.666734,0.315408,0.017858,0.351326,103
4,Tenaga Nasional,5347.KL,Tenaga Nasional Bhd (KL: TENAGA ) reported a 2...,1787848535000,816073,"Tenaga Nasional, Inari Amertron, Sunway, Berja...",https://theedgemalaysia.com/node/816073,1,2026-08-27 16:35:35+00:00,2026-08-28 00:35:35+08:00,2026-08-28,negative,0.029521,0.960594,0.009884,-0.931073,128


In [3]:
empty_text = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print("Textes vides :", empty_text.sum())

print(
    "Doublons textuels :",
    df.duplicated(subset=["text"]).sum()
)

duplicates = df[
    df.duplicated(
        subset=["company", "nid", "info_index"],
        keep=False
    )
]

print(
    "Doublons company/nid/info_index :",
    len(duplicates)
)


Textes vides : 0
Doublons textuels : 105
Doublons company/nid/info_index : 0


In [4]:
rag_df = df.copy()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    return text

rag_df["text_clean"] = rag_df["text"].apply(clean_text)

rag_df = rag_df[
    rag_df["text_clean"] != ""
].copy()

print("Nombre de lignes :", len(rag_df))


Nombre de lignes : 675


In [5]:
print(
    rag_df["n_tokens"]
    .sort_values(ascending=False)
    .head(10)
)


7      1236
115     289
54      267
76      252
539     252
538     252
36      245
116     243
89      236
274     235
Name: n_tokens, dtype: int64


In [6]:
rag_df = rag_df[
    rag_df["n_tokens"] <= 300
].copy()

print("Nombre de passages après suppression :", len(rag_df))
print("Longueur max :", rag_df["n_tokens"].max())


Nombre de passages après suppression : 674
Longueur max : 289


In [7]:
rag_df["doc_id"] = (
    rag_df["ticker"].astype(str)
    + "_"
    + rag_df["nid"].astype(str)
    + "_"
    + rag_df["info_index"].astype(str)
)

def create_text_hash(text):
    return hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()

rag_df["text_hash"] = (
    rag_df["text_clean"]
    .apply(create_text_hash)
)

rag_df.head()


,company,ticker,text,created,nid,article_title,article_url,info_index,datetime,datetime_my,date,sentiment,positive_score,negative_score,neutral_score,sentiment_score,n_tokens,text_clean,doc_id,text_hash
0,Petronas Chemicals,5183.KL,PETRONAS Chemicals Group Bhd (KL: PCHEM ) rose...,1789397619000,818088,"Kelington, NuEnergy, GFM, Exsim Hospitality, K...",https://theedgemalaysia.com/node/818088,6,2026-09-14 14:53:39+00:00,2026-09-14 22:53:39+08:00,2026-09-14,positive,0.944639,0.023755,0.031606,0.920885,114,PETRONAS Chemicals Group Bhd (KL: PCHEM ) rose...,5183.KL_818088_6,0d1e59927fceff3c20543d872c0ef9e524d2339e
1,Maybank,1155.KL,Malayan Banking Bhd (KL: MAYBANK ) has receive...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,2,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08,positive,0.561400,0.009733,0.428867,0.551667,125,Malayan Banking Bhd (KL: MAYBANK ) has receive...,1155.KL_817315_2,6e4414a0ffdb2c1bcb7bee80996bb0df487ad9dc
2,Public Bank,1295.KL,Public Bank Bhd (KL: PBBANK ) is proposing to ...,1788879258000,817315,"Public Bank, Maybank, Hektar REIT, Cuscapi, He...",https://theedgemalaysia.com/node/817315,1,2026-09-08 14:54:18+00:00,2026-09-08 22:54:18+08:00,2026-09-08,neutral,0.033205,0.019832,0.946963,0.013373,145,Public Bank Bhd (KL: PBBANK ) is proposing to ...,1295.KL_817315_1,897857803b79dbf6382db8e3db9ccd8b6bb1b265
3,CIMB,1023.KL,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,1787932174000,816234,"Yinson, Gamuda, WCT, Zetrix AI, HeiTech Padu, ...",https://theedgemalaysia.com/node/816234,6,2026-08-28 15:49:34+00:00,2026-08-28 23:49:34+08:00,2026-08-28,positive,0.666734,0.315408,0.017858,0.351326,103,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,1023.KL_816234_6,63268938a1bb0ec8a991b470e08a31befae13373
4,Tenaga Nasional,5347.KL,Tenaga Nasional Bhd (KL: TENAGA ) reported a 2...,1787848535000,816073,"Tenaga Nasional, Inari Amertron, Sunway, Berja...",https://theedgemalaysia.com/node/816073,1,2026-08-27 16:35:35+00:00,2026-08-28 00:35:35+08:00,2026-08-28,negative,0.029521,0.960594,0.009884,-0.931073,128,Tenaga Nasional Bhd (KL: TENAGA ) reported a 2...,5347.KL_816073_1,8774e94a63bf72630a6335a6bebedddb9aaac109


In [8]:
rag_df.to_csv(
    "../data/preprocessed_data/news_data_sentiment_rag.csv",
    index=False
)

print("Dataset RAG sauvegardé.")


Dataset RAG sauvegardé.


In [9]:
def row_to_document(row):

    metadata = {
        "doc_id": str(row["doc_id"]),
        "text_hash": str(row["text_hash"]),
        "company": str(row["company"]),
        "ticker": str(row["ticker"]),
        "nid": str(row["nid"]),
        "info_index": int(row["info_index"]),
        "date": str(row["date"]),
        "article_title": str(row["article_title"]),
        "article_url": str(row["article_url"]),
        "sentiment": str(row["sentiment"]),
        "sentiment_score": float(row["sentiment_score"]),
        "positive_score": float(row["positive_score"]),
        "negative_score": float(row["negative_score"]),
        "neutral_score": float(row["neutral_score"]),
        "source": "The Edge Malaysia",
        "document_type": "news"
    }

    return Document(
        page_content=row["text_clean"],
        metadata=metadata
    )


In [10]:
documents = [
    row_to_document(row)
    for _, row in rag_df.iterrows()
]

print("Nombre de documents :", len(documents))


Nombre de documents : 674


In [11]:
import pickle

with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Documents sauvegardés :", len(documents))

Documents sauvegardés : 674


In [12]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
test_embedding = embedding_model.embed_query(
    documents[0].page_content
)

print("Type :", type(test_embedding))
print("Dimension :", len(test_embedding))
print("Premières valeurs :", test_embedding[:10])


Type : <class 'list'>
Dimension : 768
Premières valeurs : [-0.011914713308215141, -0.03276510909199715, -0.01085567381232977, -0.016357099637389183, 0.04931023344397545, -0.004819060675799847, -0.01621825434267521, 0.03622579947113991, -0.009554876945912838, 0.01065667811781168]


In [14]:
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

print("Index FAISS créé.")


Index FAISS créé.


In [15]:
from sentence_transformers import CrossEncoder


# Charger le reranker
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def retrieve_and_rerank(
    query,
    k=5,
    fetch_k=20
):
    """
    1. FAISS récupère fetch_k candidats
    2. Le CrossEncoder les rerank
    3. On conserve les k meilleurs
    """

    # 1. Candidate retrieval avec FAISS
    candidates = vectorstore.similarity_search(
        query,
        k=fetch_k
    )

    # 2. Construire les couples :
    # [question, document]
    pairs = [
        [query, doc.page_content]
        for doc in candidates
    ]

    # 3. Scores du CrossEncoder
    scores = reranker.predict(
        pairs
    )

    # 4. Trier du score le plus élevé au plus faible
    ranked_indices = np.argsort(
        scores
    )[::-1]

    # 5. Top k final
    reranked_docs = [
        candidates[i]
        for i in ranked_indices[:k]
    ]

    return reranked_docs

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [16]:
# ============================================================
# 8. TEST DU RETRIEVER FINAL
# ============================================================

question = """
What were Maybank's 4QFY2025 net profit
and dividend per share?
"""

retrieved_docs = retrieve_and_rerank(
    question,
    k=5
)


for i, doc in enumerate(
    retrieved_docs,
    start=1
):

    print("=" * 100)

    print(
        f"SOURCE {i}"
    )

    print(
        "Company :",
        doc.metadata["company"]
    )

    print(
        "Date :",
        doc.metadata["date"]
    )

    print(
        "URL :",
        doc.metadata["article_url"]
    )

    print()

    print(
        doc.page_content
    )

    print()

SOURCE 1
Company : Maybank
Date : 2025-08-27
URL : https://theedgemalaysia.com/node/768283

Malayan Banking Bhd ’s (KL: MAYBANK ) net profit for 2QFY2025 rose nearly 4% y-o-y to RM2.63 billion from RM2.53 billion, as higher income offset rising provisions for bad debts. Net interest income inched up 1.1% y-o-y while non-interest income climbed 18% during the quarter. Maybank declared a first interim dividend of 30 sen per share. Maybank also reiterated its target for a return on equity of at least 11.3% for FY2025. However, Maybank lowered its loan growth guidance for the year to around 3% from 5%-6% previously, citing client caution amid tariff uncertainties, which had led to delays in planned investments and capital-raising activities. — Maybank posts 4% rise in 2Q net profit, declares 30 sen dividend

SOURCE 2
Company : Maybank
Date : 2026-02-27
URL : https://theedgemalaysia.com/node/794278

Malayan Banking Bhd ’s (KL: MAYBANK ) net profit for the fourth quarter ended Dec 31, 2025 (

In [17]:
# ============================================================
# 9. CONSTRUCTION DU CONTEXTE
# ============================================================

def build_context(documents):

    context_parts = []

    for i, doc in enumerate(
        documents,
        start=1
    ):

        metadata = doc.metadata

        context = f"""
[Source {i}]

Company: {metadata["company"]}
Ticker: {metadata["ticker"]}
Date: {metadata["date"]}
Source: {metadata["source"]}
URL: {metadata["article_url"]}

Content:
{doc.page_content}
"""

        context_parts.append(
            context.strip()
        )

    return "\n\n".join(
        context_parts
    )

In [18]:
context = build_context(
    retrieved_docs
)

print(context)

[Source 1]

Company: Maybank
Ticker: 1155.KL
Date: 2025-08-27
Source: The Edge Malaysia
URL: https://theedgemalaysia.com/node/768283

Content:
Malayan Banking Bhd ’s (KL: MAYBANK ) net profit for 2QFY2025 rose nearly 4% y-o-y to RM2.63 billion from RM2.53 billion, as higher income offset rising provisions for bad debts. Net interest income inched up 1.1% y-o-y while non-interest income climbed 18% during the quarter. Maybank declared a first interim dividend of 30 sen per share. Maybank also reiterated its target for a return on equity of at least 11.3% for FY2025. However, Maybank lowered its loan growth guidance for the year to around 3% from 5%-6% previously, citing client caution amid tariff uncertainties, which had led to delays in planned investments and capital-raising activities. — Maybank posts 4% rise in 2Q net profit, declares 30 sen dividend

[Source 2]

Company: Maybank
Ticker: 1155.KL
Date: 2026-02-27
Source: The Edge Malaysia
URL: https://theedgemalaysia.com/node/794278


In [19]:
# ============================================================
# 10. PROMPT RAG
# ============================================================

from langchain_core.prompts import ChatPromptTemplate


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a financial research assistant specialized
in Malaysian listed companies.

Answer the user's question using ONLY the information
contained in the provided context.

Rules:

1. Do not use outside knowledge.

2. Do not invent financial figures, events,
   dates or explanations.

3. If the available context is insufficient,
   explicitly say that the retrieved documents
   do not contain enough information.

4. Cite the supporting source using:
   [Source 1], [Source 2], etc.

5. Be precise and concise.

6. When several documents provide relevant information,
   combine them carefully.

7. Do not treat information from one company as
   information about another company.
"""
        ),

        (
            "human",
            """
QUESTION:

{question}


CONTEXT:

{context}


ANSWER:
"""
        )
    ]
)

In [20]:
# ============================================================
# 11. LLM
# ============================================================

import os
from getpass import getpass
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

# Demander la clé seulement si elle n'existe pas déjà
if not os.getenv("GOOGLE_API_KEY"):

    os.environ["GOOGLE_API_KEY"] = getpass(
        "Google API Key: "
    )


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [21]:
# ============================================================
# 12. GENERATION CHAIN
# ============================================================

from langchain_core.output_parsers import StrOutputParser


generation_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

In [22]:
# ============================================================
# 13. RAG COMPLET
# ============================================================

def ask_rag(
    question,
    k=5
):

    # ------------------------------------------
    # 1. Retrieval + reranking
    # ------------------------------------------

    documents = retrieve_and_rerank(
        query=question,
        k=k,
        fetch_k=20
    )


    # ------------------------------------------
    # 2. Construire le contexte
    # ------------------------------------------

    context = build_context(
        documents
    )


    # ------------------------------------------
    # 3. Génération
    # ------------------------------------------

    answer = generation_chain.invoke(
        {
            "question": question,
            "context": context
        }
    )


    return {
        "question": question,
        "answer": answer,
        "documents": documents
    }

In [23]:
# ============================================================
# 14. FIRST RAG QUERY
# ============================================================

question = """
What were Maybank's 4QFY2025 net profit
and dividend per share?
"""


result = ask_rag(
    question
)


print(
    "QUESTION:"
)

print(
    result["question"]
)


print(
    "\nANSWER:"
)

print(
    result["answer"]
)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


QUESTION:

What were Maybank's 4QFY2025 net profit
and dividend per share?


ANSWER:
Maybank's net profit for 4QFY2025 was RM2.68 billion, and a dividend of 33 sen per share was declared [Source 2].


In [24]:
# ============================================================
# 15. DISPLAY SOURCES
# ============================================================

def display_sources(documents):

    print(
        "\nSOURCES:"
    )

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\n[Source {i}]"
        )

        print(
            "Company:",
            doc.metadata["company"]
        )

        print(
            "Date:",
            doc.metadata["date"]
        )

        print(
            "URL:",
            doc.metadata["article_url"]
        )

In [25]:
# ============================================================
# 16. USER-FRIENDLY FUNCTION
# ============================================================

def ask_finagent(question):

    result = ask_rag(
        question
    )

    print(
        "\nQUESTION\n"
    )

    print(
        result["question"]
    )


    print(
        "\nANSWER\n"
    )

    print(
        result["answer"]
    )


    print(
        "\n" + "=" * 100
    )


    display_sources(
        result["documents"]
    )

In [ ]:
# ============================================================
# 17. GENERATION DES REPONSES DU BENCHMARK
# ============================================================

eval_df = pd.read_csv(
    "../data/rag_evaluation_50_questions.csv"
)

test_df = eval_df

generated_results = []


for i, row in test_df.iterrows():

    question = row["question"]

    print(
        f"Question {i + 1}/{len(eval_df)}"
    )

    result = ask_rag(
        question
    )

    generated_results.append(
        {
            "question_id":
                row["question_id"],

            "question":
                question,

            "expected_answer":
                row["expected_answer"],

            "generated_answer":
                result["answer"],

            "retrieved_doc_ids":
                "|".join(
                    [
                        doc.metadata["doc_id"]
                        for doc in result["documents"]
                    ]
                )
        }
    )

    time.sleep(15)


generation_eval_df = pd.DataFrame(
    generated_results
)


display(
    generation_eval_df.head()
)

In [ ]:
generation_eval_df.to_csv("../data/rag_generation_results_50.csv", index=False)
